In [73]:
%load_ext autoreload
%autoreload 2

from tgraasp import STx_encoder, STx_discriminator, STx_ARGVA
from tgraasp import DatasetScheme, TGraaspDataBuilder

sample_ids = ['P0', 'P1', 'P2','P3','P4']
raw_dir    = '/home/ubuntu/sda1/qiyuan/fengyang/soft_final/T-GRAASP/T-GRAASP/data/'
builder = TGraaspDataBuilder(raw_dir=raw_dir)
graph_scheme = DatasetScheme(
    counts_tmpl="{sid}_hvg_sp_data.txt",
    labels_tmpl="{sid}_meta_sp.txt",
    positions_tmpl="{sid}_position_sp.txt",
    label_col="integrated_snn_res.0.4",
    sep=r"\s+",
    with_edges=True,
    coord_cols=(1, 2), 
)
sp_graph_list, sp_adj_list = builder.build_dataset(
    sample_ids=sample_ids,
    scheme=graph_scheme,
    test_ratio=0.1,
    distance_ref_k=100,
    distance_ref_rank=8,
    distance_margin=0.0,
)

basic_scheme = DatasetScheme(
    counts_tmpl="{sid}_hvg_data.txt",
    labels_tmpl="{sid}_meta.txt",
    sep=r"\s+",
    with_edges=False,
)
sp_basic_list = builder.build_dataset(
    sample_ids=sample_ids,
    scheme=basic_scheme,
)

N_nodes   = sp_graph_list[0].x.size(0)   # 节点个数
F_inputs  = sp_graph_list[0].x.size(1)   

ppi_file     = f'{raw_dir}/PPI.connect1.txt'
connectivity = builder.load_ppi_connectivity(ppi_file, F_inputs)    # shape = [2, E]



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[build_dataset] Loading sample P0 (with_edges=True)
[build_dataset] Loading sample P1 (with_edges=True)
[build_dataset] Loading sample P2 (with_edges=True)
[build_dataset] Loading sample P3 (with_edges=True)
[build_dataset] Loading sample P4 (with_edges=True)
[build_dataset] Loading sample P0 (with_edges=False)
[build_dataset] Loading sample P1 (with_edges=False)
[build_dataset] Loading sample P2 (with_edges=False)
[build_dataset] Loading sample P3 (with_edges=False)
[build_dataset] Loading sample P4 (with_edges=False)


In [74]:
encoder = STx_encoder(
    in_channels      = F_inputs,        
    hidden_channels  = 96,
    out_channels     = 16,
    m = 18, l = 3, K = 3,
    connect = connectivity,        
    pi = 0.75,
    n_heads = 8,
)

discriminator = STx_discriminator(
    in_channels     = 16,  
    hidden_channels = 64,   
    out_channels    = 1
)

model = STx_ARGVA(
    encoder      = encoder,
    discriminator= discriminator,
    l            = 3          
)

In [ ]:
from tgraasp.train import TGraaspTrainer

trainer = TGraaspTrainer(
    model=model,
    sp_graph_list=sp_graph_list,
    sc_graph_list=sp_basic_list,
    sp_adj_list=sp_adj_list,
    F_inputs=F_inputs,          # ← 加这一行
    device='cuda',
    pretrain_epochs=2000,
)
best_model_state = trainer.pretrain()

=== TGraaspTrainer: Data Debug ===
Spatial datasets: 5
scRNA datasets:  5
Spatial adjs:    5
Start Pretrain...
[Pretrain] Epoch 001 | Dataset 00 | ARI 0.089 | ACC 0.127 | NMI 0.179 | AUC 0.491 | AP 0.550 | ACC1 0.500
[Pretrain] Epoch 001 | Dataset 01 | ARI 0.078 | ACC 0.226 | NMI 0.181 | AUC 0.507 | AP 0.560 | ACC1 0.500
[Pretrain] Epoch 001 | Dataset 02 | ARI 0.223 | ACC 0.277 | NMI 0.251 | AUC 0.507 | AP 0.536 | ACC1 0.500
[Pretrain] Epoch 001 | Dataset 03 | ARI 0.041 | ACC 0.094 | NMI 0.124 | AUC 0.508 | AP 0.525 | ACC1 0.500
[Pretrain] Epoch 001 | Dataset 04 | ARI 0.282 | ACC 0.347 | NMI 0.297 | AUC 0.499 | AP 0.546 | ACC1 0.500
✔ New best mean AUROC 0.5025 at epoch 1
Epoch 001 | Avg Loss 18.9957 | Mean AUC 0.5025 | Best 0.5025 (Epoch 1)
[Pretrain] Epoch 002 | Dataset 00 | ARI 0.125 | ACC 0.272 | NMI 0.258 | AUC 0.498 | AP 0.555 | ACC1 0.500
[Pretrain] Epoch 002 | Dataset 01 | ARI 0.148 | ACC 0.342 | NMI 0.266 | AUC 0.511 | AP 0.567 | ACC1 0.500
[Pretrain] Epoch 002 | Dataset 02 | 